Сначала импортируем необходимые библиотеки

In [ ]:
!pip install rdkit
!pip install mordred[full]

In [ ]:
!pip install tqdm

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import requests
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors
from tqdm import tqdm
from rdkit.Chem import AllChem

теперь мы загружаем из химической базы данных Chemble информацию о мишени GSK3B, которая является перспективной мишенью для лечения болезни Альцгеймера (данные загружены для Homo Sapiens)

In [ ]:
from google.colab import files
import pandas as pd

# Загружаем файл
uploaded = files.upload()

# Читаем CSV файл
df = pd.read_csv('mol2mol_similarity_new.csv')
print("Файл загружен успешно!")
df.head()

Saving mol2mol_similarity_new.csv to mol2mol_similarity_new.csv
Файл загружен успешно!


,SMILES,SMILES_state,Input_SMILES,Tanimoto,NLL
0,C[C@](N)(Cc1ccc(OCc2ccc(Cl)cc2)c(O)c1)C(=O)O,1,C[C@](N)(Cc1ccc(O)c(O)c1)C(=O)O,0.478261,4.42
1,C[C@](N)(Cc1ccc(OCc2ccccc2)c(O)c1)C(=O)O,1,C[C@](N)(Cc1ccc(O)c(O)c1)C(=O)O,0.492537,4.72
2,CC1(C)CC[C@]2(C(=O)O)CC[C@]3(C)C(=CC[C@@H]4[C@...,1,CC1(C)CC[C@]2(C(=O)OCCCCO[N+](=O)[O-])CC[C@]3(...,0.725000,1.15
3,CC1(C)CC[C@]2(C(=O)O)CC[C@]3(C)C(=CC[C@@H]4[C@...,1,CC1(C)CC[C@]2(C(=O)OCCCCO[N+](=O)[O-])CC[C@]3(...,0.725000,2.80
4,CC1(C)CC[C@]2(C(=O)O)CC[C@]3(C)C(=CC[C@@H]4[C@...,1,CC1(C)CC[C@]2(C(=O)OCCO[N+](=O)[O-])CC[C@]3(C)...,0.756757,0.90


Теперь сделаем предобработку нашего датасета

In [ ]:
# Выбор нужных колонок и очистка данных
processed_df = df[['SMILES']].copy()

# Удаление пропущенных значений
processed_df = processed_df.dropna()

# Удаление дубликатов по SMILES
processed_df = processed_df.drop_duplicates(subset=['SMILES'])
processed_df = processed_df.rename(columns={'SMILES': 'smiles'})

print(f"После предобработки осталось {len(processed_df)} уникальных соединений")
# Восстановление правильной нумерации индексов
processed_df=processed_df.reset_index(drop=True)
processed_df

После предобработки осталось 6244 уникальных соединений


,smiles
0,C[C@](N)(Cc1ccc(OCc2ccc(Cl)cc2)c(O)c1)C(=O)O
1,C[C@](N)(Cc1ccc(OCc2ccccc2)c(O)c1)C(=O)O
2,CC1(C)CC[C@]2(C(=O)O)CC[C@]3(C)C(=CC[C@@H]4[C@...
3,CC1(C)CC[C@]2(C(=O)O)CC[C@]3(C)C(=CC[C@@H]4[C@...
4,CC1(C)CC[C@]2(C(=O)O)CC[C@]3(C)C(=CC[C@@H]4[C@...
...,...
6239,Cc1ncsc1[C@H](Nc1cc(Cl)c2ncc(C#N)c(N[C@H](C)c3...
6240,Cc1nc(CNc2cc(Cl)c3ncc(C#N)c(Nc4ccc(Cl)c(Cl)c4F...
6241,Cc1nc([C@H](Nc2cc(Cl)c3ncc(C#N)c(Nc4ccc(Cl)c(C...
6242,N#Cc1cnc2c(Cl)cc(NC(C3=NNCCN3)c3cccnc3)cc2c1Nc...


теперь мы вычислим числовые молекулярные дескрипторы

In [ ]:
desc_names = [d[0] for d in Descriptors._descList]
calc = MoleculeDescriptors.MolecularDescriptorCalculator(desc_names)
desc_data = []
for mol_repr in processed_df['smiles']:
    mol = None
    # Пробуем сначала как molblock, потом как SMILES
    try:
        mol = Chem.MolFromMolBlock(mol_repr, sanitize=True)
    except Exception:
        mol = None
    if mol is None:
        try:
            mol = Chem.MolFromSmiles(mol_repr)
        except Exception:
            mol = None
    # Если молекула невалидна, дескрипторы будут NaN
    if mol is not None:
        vals = calc.CalcDescriptors(mol)
        desc_data.append(vals)
    else:
        desc_data.append([np.nan]*len(desc_names))
df_desc = pd.DataFrame(desc_data, columns=desc_names)
df_rdkit = processed_df.reset_index(drop=True).join(df_desc)
print(f'После расчета дескрипторов: {len(df_rdkit)} строк')
print(f'Колонки после RDKit-дескрипторов ({len(df_rdkit.columns)}):')
print(list(df_rdkit.columns))

После расчета дескрипторов: 6244 строк
Колонки после RDKit-дескрипторов (218):
['smiles', 'MaxAbsEStateIndex', 'MaxEStateIndex', 'MinAbsEStateIndex', 'MinEStateIndex', 'qed', 'SPS', 'MolWt', 'HeavyAtomMolWt', 'ExactMolWt', 'NumValenceElectrons', 'NumRadicalElectrons', 'MaxPartialCharge', 'MinPartialCharge', 'MaxAbsPartialCharge', 'MinAbsPartialCharge', 'FpDensityMorgan1', 'FpDensityMorgan2', 'FpDensityMorgan3', 'BCUT2D_MWHI', 'BCUT2D_MWLOW', 'BCUT2D_CHGHI', 'BCUT2D_CHGLO', 'BCUT2D_LOGPHI', 'BCUT2D_LOGPLOW', 'BCUT2D_MRHI', 'BCUT2D_MRLOW', 'AvgIpc', 'BalabanJ', 'BertzCT', 'Chi0', 'Chi0n', 'Chi0v', 'Chi1', 'Chi1n', 'Chi1v', 'Chi2n', 'Chi2v', 'Chi3n', 'Chi3v', 'Chi4n', 'Chi4v', 'HallKierAlpha', 'Ipc', 'Kappa1', 'Kappa2', 'Kappa3', 'LabuteASA', 'PEOE_VSA1', 'PEOE_VSA10', 'PEOE_VSA11', 'PEOE_VSA12', 'PEOE_VSA13', 'PEOE_VSA14', 'PEOE_VSA2', 'PEOE_VSA3', 'PEOE_VSA4', 'PEOE_VSA5', 'PEOE_VSA6', 'PEOE_VSA7', 'PEOE_VSA8', 'PEOE_VSA9', 'SMR_VSA1', 'SMR_VSA10', 'SMR_VSA2', 'SMR_VSA3', 'SMR_VSA4', 'S

In [ ]:
#  Создаем датафрейм только с дескрипторами без информации об активности (pValue) и структуре молекулы (smiles)
df_descriptors_only = df_rdkit.drop(columns=['smiles'], errors='ignore').copy()
df_descriptors_only.head()

# Удаляем нечисловые признаки для датафрейма, состоящего из одних дескрипторов
df_descriptors_only = df_descriptors_only.select_dtypes(include=[np.number])

print(f'Количество строк после удаления нечисловых признаков: {df_descriptors_only.shape[0]}')
print(f'Количество столбцов после удаления нечисловых признаков: {df_descriptors_only.shape[1]}')

# Удаляем признаки, где есть хотя бы один inf, -inf, NaN или экстремум в датафрейме из одних дескрипторов
extreme = 1e6
df_descriptors_only = df_descriptors_only.loc[:, (np.isfinite(df_descriptors_only).all(axis=0)) &
                                                (df_descriptors_only.abs() < extreme).all(axis=0)]



Количество строк после удаления нечисловых признаков: 6244
Количество столбцов после удаления нечисловых признаков: 217


In [ ]:
# Удаляем константные признаки, выводим результаты
selector = VarianceThreshold(threshold=0.01)
X_var = selector.fit_transform(df_descriptors_only)
selected_cols = df_descriptors_only.columns[selector.get_support()]
df_descriptors_only = pd.DataFrame(X_var, columns=selected_cols, index=df_descriptors_only.index)

print(f'Количество строк после удаления констант: {df_descriptors_only.shape[0]}')
print(f'Количество столбцов после удаления констант: {df_descriptors_only.shape[1]}')

# Удаляем высоко коррелированные признаки, выводим результаты
corr = df_descriptors_only.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
df_descriptors_only = df_descriptors_only.drop(columns=to_drop)

print(f'Количество строк после удаления высококор.: {df_descriptors_only.shape[0]}')
print(f'Количество столбцов после удаления высококор.: {df_descriptors_only.shape[1]}')

Количество строк после удаления констант: 6244
Количество столбцов после удаления констант: 180
Количество строк после удаления высококор.: 6244
Количество столбцов после удаления высококор.: 151


In [ ]:
df_final = pd.concat(
    (df_rdkit[["smiles"]], df_descriptors_only),
    axis=1
)

df_final.head()

,smiles,MaxAbsEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,FpDensityMorgan1,FpDensityMorgan2,FpDensityMorgan3,...,fr_priamide,fr_pyridine,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_thiazole,fr_thiophene,fr_unbrch_alkane,fr_urea
0,C[C@](N)(Cc1ccc(OCc2ccc(Cl)cc2)c(O)c1)C(=O)O,11.038991,0.057729,-1.390753,0.754327,13.347826,335.787,1.217391,1.826087,2.391304,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,C[C@](N)(Cc1ccc(OCc2ccccc2)c(O)c1)C(=O)O,11.024450,0.032759,-1.376138,0.761117,13.363636,301.342,1.181818,1.863636,2.454545,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,CC1(C)CC[C@]2(C(=O)O)CC[C@]3(C)C(=CC[C@@H]4[C@...,12.762619,0.020449,-0.587725,0.417821,54.058824,472.710,0.941176,1.617647,2.264706,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,CC1(C)CC[C@]2(C(=O)O)CC[C@]3(C)C(=CC[C@@H]4[C@...,12.762619,0.020449,-0.587725,0.417821,54.058824,472.710,0.941176,1.617647,2.264706,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,CC1(C)CC[C@]2(C(=O)O)CC[C@]3(C)C(=CC[C@@H]4[C@...,12.753229,0.030113,-0.553469,0.409055,52.424242,456.711,0.878788,1.545455,2.212121,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
df_final.to_csv("data2m2m.csv")